# Upload a PDF to Deka Box (Cloudeka's S3-compatible storage)

**Deka Box** is Cloudeka's object storage. It speaks the **S3 API**, so we can use the standard AWS SDK for Python, **`boto3`**, and just point it at the Deka Box endpoint instead of AWS.

In this notebook we upload a **PDF document** to a bucket:

1. Configure credentials and the Deka Box endpoint.
2. Build an S3 client for Deka Box.
3. Upload a PDF into a bucket.
4. Verify it landed, and generate a link to it.

Later RAG notebooks read source PDFs from a bucket like this one.

## Dependencies

This notebook needs `boto3` (the S3 client) and `python-dotenv` (loads credentials from a `.env` file so they stay out of the notebook). They are listed in `../requirements.txt`; install once from the `day2skk` folder:

```bash
pip install -r requirements.txt
```

## Configuration

Get your **Access Key** and **Secret Key** and the **endpoint** from the Deka Box console (S3 credentials / access keys section), then create a file named `.env` next to this notebook:

```dotenv
ACCESS_KEY_ID=your-access-key
SECRET_ACCESS_KEY=your-secret-key
S3_ENDPOINT_URL=https://your-deka-box-endpoint   # from the Deka Box console
REGION=us-east-1                                 # any value works for most S3-compatible stores
S3_BUCKET=my-bucket                              # an existing bucket you can write to
```

In production, never commit real keys. Keep `.env` in your `.gitignore`.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("var.env")   # loads variables from a local .env file, if present

ACCESS_KEY_ID = os.environ.get("ACCESS_KEY_ID")
SECRET_ACCESS_KEY = os.environ.get("SECRET_ACCESS_KEY")
S3_ENDPOINT_URL = os.environ.get("S3_ENDPOINT_URL")
REGION = os.environ.get("REGION", "us-east-1")
S3_BUCKET = os.environ.get("S3_BUCKET")

# Sanity check without printing the secrets themselves.
print("ACCESS_KEY_ID set:    ", bool(ACCESS_KEY_ID))
print("SECRET_ACCESS_KEY set:", bool(SECRET_ACCESS_KEY))
print("S3_ENDPOINT_URL:      ", S3_ENDPOINT_URL)
print("REGION:               ", REGION)
print("S3_BUCKET:            ", S3_BUCKET)

## Build the Deka Box client

This is a normal `boto3` S3 client with two Deka-specific touches:

- **`endpoint_url`** points at Deka Box instead of AWS.
- **The checksum `Config`**: recent `botocore` (>= 1.36) adds a default CRC32 checksum that forces `aws-chunked` / `Transfer-Encoding: chunked`, which some S3-compatible providers (Cloudeka/Ceph/MinIO) reject with `NotImplemented`. Setting the checksum behaviour to `when_required` avoids that.

In [ ]:
import boto3
from botocore.config import Config


def make_client():
    """Build an S3 client pointed at Deka Box."""
    return boto3.client(
        "s3",
        region_name=REGION,
        endpoint_url=S3_ENDPOINT_URL or None,
        aws_access_key_id=ACCESS_KEY_ID,
        aws_secret_access_key=SECRET_ACCESS_KEY,
        config=Config(
            request_checksum_calculation="when_required",
            response_checksum_validation="when_required",
        ),
    )


client = make_client()
print("client ready for endpoint:", client.meta.endpoint_url)

## Create a sample PDF to upload

So the notebook runs end-to-end, we generate a small but valid one-page PDF using no external libraries. Replace it with your own PDF (a report, a manual, a paper, ...) when using it for real.

In [ ]:
from pathlib import Path


def make_sample_pdf(path: Path, text: str = "Hello from Deka Box") -> None:
    """Write a minimal but valid one-page PDF (no external libraries).

    Byte offsets for the cross-reference table are computed as we go, so the
    resulting file is a real PDF that PDF readers can open.
    """
    objects = [
        b"<< /Type /Catalog /Pages 2 0 R >>",
        b"<< /Type /Pages /Kids [3 0 R] /Count 1 >>",
        b"<< /Type /Page /Parent 2 0 R /MediaBox [0 0 612 792] "
        b"/Contents 4 0 R /Resources << /Font << /F1 5 0 R >> >> >>",
    ]
    stream = b"BT /F1 24 Tf 72 700 Td (" + text.encode("latin-1") + b") Tj ET"
    objects.append(b"<< /Length %d >>\nstream\n" % len(stream) + stream + b"\nendstream")
    objects.append(b"<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica >>")

    pdf = bytearray(b"%PDF-1.4\n")
    offsets = []
    for i, obj in enumerate(objects, start=1):
        offsets.append(len(pdf))
        pdf += b"%d 0 obj\n" % i + obj + b"\nendobj\n"

    xref_pos = len(pdf)
    count = len(objects) + 1
    pdf += b"xref\n0 %d\n" % count
    pdf += b"0000000000 65535 f \n"
    for off in offsets:
        pdf += b"%010d 00000 n \n" % off
    pdf += b"trailer\n<< /Size %d /Root 1 0 R >>\nstartxref\n%d\n%%%%EOF" % (count, xref_pos)
    path.write_bytes(bytes(pdf))


file_path = Path("sample.pdf")
make_sample_pdf(file_path)

# The object key is its path/name inside the bucket. Use a prefix to group files.
object_key = f"uploads/{file_path.name}"
print(f"created {file_path} ({file_path.stat().st_size} bytes)")
print(f"will upload {file_path}  ->  s3://{S3_BUCKET}/{object_key}")

## Upload the PDF

We use **`put_object`** with the file read as **bytes**, rather than the higher-level `upload_file`. `upload_file` streams with `Transfer-Encoding: chunked`, which some S3-compatible providers reject. Sending bytes gives a known `Content-Length` and avoids chunked encoding - the reliable choice for Deka Box.

We also pass **`ContentType="application/pdf"`** so Deka Box stores and serves the object with the correct type (e.g. a browser opening the presigned link renders the PDF instead of downloading it as unknown data).

In [ ]:
from botocore.exceptions import BotoCoreError, ClientError


def upload(client, bucket: str, file_path: Path, key: str,
           content_type: str = "application/pdf") -> None:
    print(f"Uploading {file_path} -> s3://{bucket}/{key}")
    with open(file_path, "rb") as f:
        client.put_object(Bucket=bucket, Key=key, Body=f.read(), ContentType=content_type)
    print("  upload OK")


if not S3_BUCKET:
    raise ValueError("S3_BUCKET is not set. Add it to your .env file.")

try:
    upload(client, S3_BUCKET, file_path, object_key)
except (BotoCoreError, ClientError) as exc:
    print("S3 error:", exc)

## Verify the upload

`head_object` fetches just the metadata of the object (size, last-modified) to confirm it exists, and `list_objects_v2` shows what is under our `uploads/` prefix.

In [ ]:
head = client.head_object(Bucket=S3_BUCKET, Key=object_key)
print("exists:", object_key)
print("  size (bytes):", head["ContentLength"])
print("  last modified:", head["LastModified"])

print("\nObjects under 'uploads/':")
listing = client.list_objects_v2(Bucket=S3_BUCKET, Prefix="uploads/")
for obj in listing.get("Contents", []):
    print(f"  {obj['Key']}  ({obj['Size']} bytes)")

## Bonus: a temporary shareable link

A **presigned URL** is a time-limited link that lets someone download a private object without needing credentials. Handy for sharing an uploaded document, or for a downstream service to fetch it.

In [ ]:
url = client.generate_presigned_url(
    "get_object",
    Params={"Bucket": S3_BUCKET, "Key": object_key},
    ExpiresIn=3600,   # valid for 1 hour
)
print("Download link (valid 1 hour):")
print(url)

## Recap

- **Deka Box is S3-compatible**, so `boto3` works once you set `endpoint_url` to the Deka Box endpoint and pass your access/secret keys.
- Keep credentials in a **`.env`** file, not in the notebook.
- Use the **checksum `Config`** (`when_required`) so newer `botocore` does not force chunked encoding that Cloudeka can reject.
- Upload with **`put_object(Body=bytes, ContentType="application/pdf")`** rather than `upload_file`, to avoid chunked `Transfer-Encoding` and tag the object as a PDF.
- Verify with **`head_object`** / **`list_objects_v2`**, and share with a **presigned URL**.

The same pattern uploads the source PDFs that the RAG pipeline will ingest.